In [1]:
from pathlib import Path

from src.data.load import leer
from src.data.limpiar import limpiar_ventas
from src.features.gmm_features import rfm
from src.segmentation.gmm import segmentar

print("Carpeta de trabajo:", Path.cwd())
print("Los cuatro módulos se importaron correctamente.")

Carpeta de trabajo: c:\Users\misae\Documents\Analisis de datos\RETO\6-ML\SVD_Pipeline
Los cuatro módulos se importaron correctamente.


In [2]:
ventas_por_archivo = {
    "sales_2024.csv": leer("data/raw/sales_2024.csv"),
    "sales.csv": leer("data/raw/sales.csv"),
}
productos = leer("data/raw/products.csv")

ventas = limpiar_ventas(ventas_por_archivo, productos)

print(f"Filas de ventas limpias: {len(ventas):,}")
display(ventas.head())

sales_2024.csv -> {'cliente': 'client_id', 'producto': 'product_id', 'fecha': 'date', 'cantidad': 'units_sold', 'monto': 'sales_amount', 'transaccion': 'transaction_id'}
sales.csv -> {'cliente': 'client_id', 'producto': 'product_id', 'fecha': 'date', 'cantidad': 'units_sold', 'monto': 'sales_amount', 'transaccion': 'transaction_id'}
Filas de ventas limpias: 2,244,587


,cliente_id,product_id,fecha,cantidad,monto,transaccion,archivo
0,23427,53,2024-10-25,1.0,53.89,T20241025_0000001,sales_2024.csv
1,23427,92,2024-10-25,1.0,34.61,T20241025_0000001,sales_2024.csv
2,23427,120,2024-10-25,2.0,160.90,T20241025_0000001,sales_2024.csv
3,23427,31,2024-10-25,3.0,200.73,T20241025_0000001,sales_2024.csv
4,23427,29,2024-10-25,1.0,62.69,T20241025_0000001,sales_2024.csv


In [3]:
import pandas as pd

corte = pd.Timestamp("2025-06-24")
hist = ventas.loc[ventas["fecha"] < corte].copy()

rfm_clientes = rfm(hist, corte)
clientes_gmm, gmm, scaler, X = segmentar(rfm_clientes)

segmentos = clientes_gmm.reset_index().assign(fecha_corte=corte)

display(segmentos[["cliente_id", "segmento", "prob_segmento"]].head())
display(segmentos["segmento"].value_counts())

,cliente_id,segmento,prob_segmento
0,1,Regulares,0.987254
1,2,Regulares,0.930159
2,3,Regulares,0.766283
3,4,Ocasionales,0.981961
4,5,Ocasionales,0.672240


segmento
Regulares      17414
Ocasionales    14174
VIP             6829
Inactivos       6564
Name: count, dtype: int64

In [4]:
hist_segmentado = hist.merge(
    segmentos[["cliente_id", "segmento"]],
    on="cliente_id",
    how="left",
    validate="many_to_one",
)

print(f"Compras antes de unir: {len(hist):,}")
print(f"Compras después de unir: {len(hist_segmentado):,}")
print(f"Compras sin segmento: {hist_segmentado['segmento'].isna().sum():,}")

display(hist_segmentado[["cliente_id", "product_id", "segmento"]].head())

Compras antes de unir: 1,583,517
Compras después de unir: 1,583,517
Compras sin segmento: 0


,cliente_id,product_id,segmento
0,23427,53,VIP
1,23427,92,VIP
2,23427,120,VIP
3,23427,31,VIP
4,23427,29,VIP


In [5]:
from src.features.interactions import construir_matriz_interacciones

matriz_global = construir_matriz_interacciones(
    ventas_hist=hist,
    clientes=segmentos["cliente_id"],
    productos=sorted(productos["product_id"].unique()),
)

print(f"Clientes: {matriz_global.shape[0]:,}")
print(f"Productos: {matriz_global.shape[1]:,}")
display(matriz_global.iloc[:5, :6])

Clientes: 44,981
Productos: 130


product_id,1,2,3,4,5,6
cliente_id,,,,,,
1,0.0,0.000000,1.098612,0.0,0.0,0.000000
2,0.0,0.000000,0.000000,0.0,0.0,0.693147
3,0.0,0.000000,0.000000,0.0,0.0,0.000000
4,0.0,0.693147,0.000000,0.0,0.0,0.000000
5,0.0,0.000000,0.000000,0.0,0.0,0.000000


In [6]:
from src.recommenders.svd import calcular_puntajes_svd

puntajes_globales = calcular_puntajes_svd(matriz_global, n_factores=30)

print("Dimensiones de los puntajes:", puntajes_globales.shape)
display(puntajes_globales.iloc[:5, :6])

Dimensiones de los puntajes: (44981, 130)


product_id,1,2,3,4,5,6
cliente_id,,,,,,
1,0.135097,0.085436,0.120453,0.151897,0.120288,0.175536
2,0.394818,0.289621,0.168428,0.222772,0.126220,0.213197
3,-0.001494,0.070779,0.147550,0.122954,0.130226,0.088885
4,0.259483,0.072395,0.168467,0.113762,0.074748,0.047988
5,0.176330,0.116931,0.025757,0.059373,0.038055,0.034554


In [7]:
puntajes_por_grupo = {}

for nombre_segmento, grupo in segmentos.groupby("segmento"):
    matriz_grupo = matriz_global.loc[grupo["cliente_id"]]

    puntajes_por_grupo[nombre_segmento] = calcular_puntajes_svd(
        matriz_grupo,
        n_factores=30,
    )

    print(f"{nombre_segmento}: {len(matriz_grupo):,} clientes")

puntajes_segmentados = pd.concat(puntajes_por_grupo.values()).loc[
    matriz_global.index
]

print("Dimensiones de los puntajes:", puntajes_segmentados.shape)
display(puntajes_segmentados.iloc[:5, :6])

Inactivos: 6,564 clientes
Ocasionales: 14,174 clientes
Regulares: 17,414 clientes
VIP: 6,829 clientes
Dimensiones de los puntajes: (44981, 130)


product_id,1,2,3,4,5,6
cliente_id,,,,,,
1,0.068418,0.084546,0.164252,0.116770,0.154671,0.169880
2,0.247485,0.229943,0.138758,0.176081,0.077137,0.223566
3,-0.025862,0.067095,0.146608,0.159789,0.115295,0.054993
4,0.345351,0.334379,0.097636,0.058580,0.131783,0.031798
5,0.053547,0.187436,-0.071874,-0.005213,0.046919,0.058301


In [10]:
from src.evaluation.metrics import evaluar_top3

fin_ventana = corte + pd.Timedelta(days=30)
ventas_futuras = ventas.loc[
    (ventas["fecha"] >= corte) & (ventas["fecha"] < fin_ventana)
]

compras_reales = construir_matriz_interacciones(
    ventas_hist=ventas_futuras,
    clientes=matriz_global.index,
    productos=matriz_global.columns,
).gt(0)

resultados, clientes_evaluados = evaluar_top3(
    compras_reales,
    {
        "Historial personal": matriz_global,
        "SVD global": puntajes_globales,
        "SVD por segmento": puntajes_segmentados,
    },
)

print(f"Clientes evaluados: {clientes_evaluados:,}")
display(resultados.round(4))

Clientes evaluados: 23,618


,Precisión@3,Recall@3,F1@3
Modelo,,,
Historial personal,0.1250,0.0949,0.0982
SVD global,0.1317,0.1008,0.1045
SVD por segmento,0.1322,0.1020,0.1053


In [21]:
cortes_evaluacion = pd.to_datetime([
    "2025-06-24",
    "2025-07-24",
    "2025-08-23",
    "2025-09-22",
    "2025-10-22",
])


def evaluar_corte(
    fecha_corte,
    factores_segmentado=30,
    peso_popularidad=0.25,
    dias_prueba=30,
):
    ventana = pd.Timedelta(days=dias_prueba)
    historial = ventas.loc[ventas["fecha"] < fecha_corte]
    futuro = ventas.loc[
        (ventas["fecha"] >= fecha_corte)
        & (ventas["fecha"] < fecha_corte + ventana)
    ]

    clientes_con_segmento, _, _, _ = segmentar(rfm(historial, fecha_corte))
    matriz = construir_matriz_interacciones(
        historial,
        clientes_con_segmento.index,
        sorted(productos["product_id"].unique()),
    )

    svd_global = calcular_puntajes_svd(matriz, n_factores=30)

    puntajes_grupos = []
    for _, grupo in clientes_con_segmento.groupby("segmento"):
        puntajes_grupos.append(
            calcular_puntajes_svd(
                matriz.loc[grupo.index],
                n_factores=factores_segmentado,
            )
        )
    svd_segmentado = pd.concat(puntajes_grupos).loc[matriz.index]
    rango_svd_segmentado = svd_segmentado.rank(axis=1, pct=True)
    popularidad_global = matriz.gt(0).sum(axis=0)
    rango_popularidad_global = popularidad_global.rank(pct=True)

    svd_segmentado_con_pop_global = (
        0.75 * rango_svd_segmentado + 0.25 * rango_popularidad_global
    )
    hibrido_segmentado = rango_svd_segmentado.copy()
    rango_svd_global = svd_global.rank(axis=1, pct=True)
    hibrido_global = rango_svd_global.copy()
    popularidad_segmentada = pd.DataFrame(
        index=matriz.index,
        columns=matriz.columns,
        dtype=float,
    )

    for _, grupo in clientes_con_segmento.groupby("segmento"):
        ids = grupo.index
        popularidad = matriz.loc[ids].gt(0).sum(axis=0)
        rango_popularidad = popularidad.rank(pct=True)
        popularidad_segmentada.loc[ids] = rango_popularidad.to_numpy()

        hibrido_segmentado.loc[ids] = (
            (1 - peso_popularidad) * rango_svd_segmentado.loc[ids]
            + peso_popularidad * rango_popularidad
        )
        hibrido_global.loc[ids] = (
            0.75 * rango_svd_global.loc[ids] + 0.25 * rango_popularidad
        )

    rango_historial = matriz.rank(axis=1, pct=True)
    rango_svd = svd_global.rank(axis=1, pct=True)
    svd_con_historial = 0.5 * rango_historial + 0.5 * rango_svd

    reales = construir_matriz_interacciones(
        futuro, matriz.index, matriz.columns
    ).gt(0)

    tabla, n_clientes = evaluar_top3(
        reales,
        {
            "Historial personal": matriz,
            "SVD global": svd_global,
            "SVD por segmento": svd_segmentado,
            "SVD por segmento + popularidad": hibrido_segmentado,
            "SVD por segmento + popularidad global": svd_segmentado_con_pop_global,
            "SVD global + popularidad por segmento": hibrido_global,
            "Popularidad por segmento": popularidad_segmentada,
            "SVD + historial": svd_con_historial,
        },
    )
    return tabla.reset_index().assign(
        Corte=fecha_corte.date(),
        Clientes_evaluados=n_clientes,
    )


tablas = []
for fecha_corte in cortes_evaluacion:
    print(f"Evaluando {fecha_corte.date()}...")
    tablas.append(evaluar_corte(fecha_corte))

resultados_por_corte = pd.concat(tablas, ignore_index=True)

f1_por_corte = resultados_por_corte.pivot(
    index="Corte", columns="Modelo", values="F1@3"
)
f1_por_corte["Ventaja segmentado vs. global"] = (
    f1_por_corte["SVD por segmento"] - f1_por_corte["SVD global"]
)

display(f1_por_corte.round(4))

promedios = resultados_por_corte.groupby("Modelo")[
    ["Precisión@3", "Recall@3", "F1@3"]
].mean()
display(promedios.sort_values("F1@3", ascending=False).round(4))

Evaluando 2025-06-24...
Evaluando 2025-07-24...
Evaluando 2025-08-23...
Evaluando 2025-09-22...
Evaluando 2025-10-22...


Modelo,Historial personal,Popularidad por segmento,SVD + historial,SVD global,SVD global + popularidad por segmento,SVD por segmento,SVD por segmento + popularidad,SVD por segmento + popularidad global,Ventaja segmentado vs. global
Corte,,,,,,,,,
2025-06-24,0.0982,0.0362,0.0969,0.1045,0.1111,0.1053,0.1119,0.1112,0.0008
2025-07-24,0.0967,0.0368,0.0901,0.0980,0.1063,0.0972,0.1059,0.1051,-0.0008
2025-08-23,0.0959,0.0367,0.0870,0.0920,0.1001,0.0921,0.1010,0.1005,0.0001
2025-09-22,0.0959,0.0372,0.0848,0.0876,0.0955,0.0876,0.0968,0.0959,0.0000
2025-10-22,0.0974,0.0380,0.0863,0.0859,0.0972,0.0865,0.0976,0.0963,0.0006


,Precisión@3,Recall@3,F1@3
Modelo,,,
SVD por segmento + popularidad,0.1291,0.0995,0.1026
SVD global + popularidad por segmento,0.1285,0.0988,0.1020
SVD por segmento + popularidad global,0.1283,0.0985,0.1018
Historial personal,0.1233,0.0933,0.0968
SVD por segmento,0.1179,0.0910,0.0937
SVD global,0.1179,0.0907,0.0936
SVD + historial,0.1125,0.0863,0.0890
Popularidad por segmento,0.0481,0.0351,0.0370


In [19]:
corte_mayo = pd.Timestamp("2025-05-25")
nombre_hibrido = "SVD por segmento + popularidad"

comparacion_mayo = pd.DataFrame({
    "Original (30 factores, 25%)": (
        evaluar_corte(corte_mayo, factores_segmentado=30, peso_popularidad=0.25)
        .set_index("Modelo")
        .loc[nombre_hibrido, ["Precisión@3", "Recall@3", "F1@3"]]
    ),
    "Elegida en abril (40 factores, 40%)": (
        evaluar_corte(corte_mayo, factores_segmentado=40, peso_popularidad=0.40)
        .set_index("Modelo")
        .loc[nombre_hibrido, ["Precisión@3", "Recall@3", "F1@3"]]
    ),
}).T

display(comparacion_mayo.round(4))

,Precisión@3,Recall@3,F1@3
"Original (30 factores, 25%)",0.143744,0.111083,0.114421
"Elegida en abril (40 factores, 40%)",0.148169,0.11459,0.118004


In [20]:
nombre_hibrido = "SVD por segmento + popularidad"
metricas_top3 = ["Precisión@3", "Recall@3", "F1@3"]

original = (
    resultados_por_corte.loc[
        resultados_por_corte["Modelo"] == nombre_hibrido
    ]
    .set_index("Corte")[metricas_top3]
    .sort_index()
)

tablas_ajustadas = []
for fecha_corte in cortes_evaluacion:
    print(f"Evaluando 40 factores y 40% en {fecha_corte.date()}...")
    tabla = evaluar_corte(
        fecha_corte,
        factores_segmentado=40,
        peso_popularidad=0.40,
    )
    tablas_ajustadas.append(tabla.loc[tabla["Modelo"] == nombre_hibrido])

ajustado = (
    pd.concat(tablas_ajustadas, ignore_index=True)
    .set_index("Corte")[metricas_top3]
    .sort_index()
)

comparacion_cortes = original[["F1@3"]].rename(
    columns={"F1@3": "Original (30 factores, 25%)"}
).join(
    ajustado[["F1@3"]].rename(
        columns={"F1@3": "Ajustado (40 factores, 40%)"}
    )
)

comparacion_cortes["Diferencia F1@3"] = (
    comparacion_cortes["Ajustado (40 factores, 40%)"]
    - comparacion_cortes["Original (30 factores, 25%)"]
)

display(comparacion_cortes.round(4))

promedios_configuraciones = pd.DataFrame({
    "Original (30 factores, 25%)": original.mean(),
    "Ajustado (40 factores, 40%)": ajustado.mean(),
}).T
display(promedios_configuraciones.round(4))

ganados = (comparacion_cortes["Diferencia F1@3"] > 0).sum()
print(f"El ajustado ganó {ganados} de {len(comparacion_cortes)} cortes.")

Evaluando 40 factores y 40% en 2025-06-24...
Evaluando 40 factores y 40% en 2025-07-24...
Evaluando 40 factores y 40% en 2025-08-23...
Evaluando 40 factores y 40% en 2025-09-22...
Evaluando 40 factores y 40% en 2025-10-22...


,"Original (30 factores, 25%)","Ajustado (40 factores, 40%)",Diferencia F1@3
Corte,,,
2025-06-24,0.1119,0.1159,0.0040
2025-07-24,0.1059,0.1115,0.0056
2025-08-23,0.1010,0.1049,0.0039
2025-09-22,0.0968,0.1010,0.0042
2025-10-22,0.0976,0.1039,0.0063


,Precisión@3,Recall@3,F1@3
"Original (30 factores, 25%)",0.1291,0.0995,0.1026
"Ajustado (40 factores, 40%)",0.1349,0.1042,0.1074


El ajustado ganó 5 de 5 cortes.


In [17]:
corte_validacion = pd.Timestamp("2025-04-25")
fin_validacion = corte_validacion + pd.Timedelta(days=30)

hist_validacion = ventas.loc[ventas["fecha"] < corte_validacion]
futuro_validacion = ventas.loc[
    (ventas["fecha"] >= corte_validacion)
    & (ventas["fecha"] < fin_validacion)
]

clientes_validacion, _, _, _ = segmentar(
    rfm(hist_validacion, corte_validacion)
)

matriz_validacion = construir_matriz_interacciones(
    hist_validacion,
    clientes_validacion.index,
    sorted(productos["product_id"].unique()),
)
reales_validacion = construir_matriz_interacciones(
    futuro_validacion,
    matriz_validacion.index,
    matriz_validacion.columns,
).gt(0)

grupos_validacion = [
    grupo.index for _, grupo in clientes_validacion.groupby("segmento")
]

# La popularidad se calcula una sola vez con las compras previas al corte.
rangos_popularidad = pd.DataFrame(
    index=matriz_validacion.index,
    columns=matriz_validacion.columns,
    dtype=float,
)

for ids in grupos_validacion:
    popularidad = matriz_validacion.loc[ids].gt(0).sum(axis=0)
    rangos_popularidad.loc[ids] = popularidad.rank(pct=True).to_numpy()

factores_a_probar = [5, 10, 15, 20, 30, 40]
pesos_a_probar = [0, 0.10, 0.25, 0.40, 0.50]
filas_ajuste = []

for n_factores in factores_a_probar:
    print(f"Evaluando {n_factores} factores...")

    # Entrena una vez por grupo y reutiliza los puntajes para los cinco pesos.
    puntajes = pd.concat([
        calcular_puntajes_svd(matriz_validacion.loc[ids], n_factores)
        for ids in grupos_validacion
    ]).loc[matriz_validacion.index]
    rangos_svd = puntajes.rank(axis=1, pct=True)

    for peso in pesos_a_probar:
        puntajes_hibridos = (
            (1 - peso) * rangos_svd + peso * rangos_popularidad
        )
        metricas, n_evaluados = evaluar_top3(
            reales_validacion, {"Híbrido": puntajes_hibridos}
        )

        filas_ajuste.append({
            "Factores": n_factores,
            "Peso popularidad": peso,
            **metricas.loc["Híbrido"].to_dict(),
        })

ajuste = pd.DataFrame(filas_ajuste).sort_values("F1@3", ascending=False)

print(f"Clientes evaluados: {n_evaluados:,}")
print("Cinco mejores combinaciones")
display(ajuste.head(5).round(4))

print("Configuración actual: 30 factores y 25% popularidad")
display(
    ajuste.loc[
        (ajuste["Factores"] == 30) & (ajuste["Peso popularidad"] == 0.25)
    ].round(4)
)

Evaluando 5 factores...
Evaluando 10 factores...
Evaluando 15 factores...
Evaluando 20 factores...
Evaluando 30 factores...
Evaluando 40 factores...
Clientes evaluados: 22,384
Cinco mejores combinaciones


,Factores,Peso popularidad,Precisión@3,Recall@3,F1@3
28,40,0.40,0.1542,0.1219,0.1248
27,40,0.25,0.1531,0.1219,0.1244
29,40,0.50,0.1534,0.1207,0.1238
22,30,0.25,0.1496,0.1187,0.1213
21,30,0.10,0.1492,0.1183,0.1209


Configuración actual: 30 factores y 25% popularidad


,Factores,Peso popularidad,Precisión@3,Recall@3,F1@3
22,30,0.25,0.1496,0.1187,0.1213


In [22]:
# Busca el cambio de día que deja el porcentaje más cercano a 80%.
registros_por_dia = ventas.groupby(ventas["fecha"].dt.normalize()).size()
registros_antes_del_dia = registros_por_dia.cumsum().shift(fill_value=0)

candidatos = registros_antes_del_dia.iloc[1:]
corte_80_20 = (candidatos / len(ventas) - 0.80).abs().idxmin()

n_entrenamiento = int(registros_antes_del_dia.loc[corte_80_20])
n_prueba = len(ventas) - n_entrenamiento
ultimo_dia = registros_por_dia.index.max()
dias_prueba = (ultimo_dia - corte_80_20).days + 1

print(f"Corte: {corte_80_20.date()}")
print(f"Entrenamiento: {n_entrenamiento:,} registros ({n_entrenamiento / len(ventas):.2%})")
print(f"Prueba: {n_prueba:,} registros ({n_prueba / len(ventas):.2%})")
print(f"Periodo de prueba: {corte_80_20.date()} a {ultimo_dia.date()} ({dias_prueba} días)")

nombre_hibrido = "SVD por segmento + popularidad"
metricas = ["Precisión@3", "Recall@3", "F1@3"]
filas = []

for nombre, factores, peso in [
    ("Original (30 factores, 25%)", 30, 0.25),
    ("Ajustado (40 factores, 40%)", 40, 0.40),
]:
    tabla = evaluar_corte(
        corte_80_20,
        factores_segmentado=factores,
        peso_popularidad=peso,
        dias_prueba=dias_prueba,
    )
    resultado = tabla.set_index("Modelo").loc[nombre_hibrido, metricas]
    filas.append(resultado.rename(nombre))

display(pd.DataFrame(filas).round(4))

Corte: 2025-08-26
Entrenamiento: 1,795,538 registros (79.99%)
Prueba: 449,049 registros (20.01%)
Periodo de prueba: 2025-08-26 a 2025-12-31 (128 días)


,Precisión@3,Recall@3,F1@3
"Original (30 factores, 25%)",0.2968,0.1007,0.1391
"Ajustado (40 factores, 40%)",0.3137,0.1075,0.1481


In [23]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KDTree

USAR_CORTE_80_20 = True

if USAR_CORTE_80_20:
    corte_exportacion = corte_80_20
    archivo = "recomendaciones_danustore_svd_80_20.xlsx"
else:
    corte_exportacion = ventas["fecha"].max().normalize() + pd.Timedelta(days=1)
    archivo = "recomendaciones_danustore_svd_historial_completo.xlsx"

historial = ventas.loc[ventas["fecha"] < corte_exportacion]
clientes_gmm, _, _, X_rfm = segmentar(rfm(historial, corte_exportacion))

catalogo = productos.set_index("product_id").sort_index()
matriz = construir_matriz_interacciones(
    historial, clientes_gmm.index, catalogo.index
)

# Mismo híbrido evaluado: 40 factores, 60% SVD y 40% popularidad.
puntajes = pd.DataFrame(index=matriz.index, columns=matriz.columns, dtype=float)

for _, grupo in clientes_gmm.groupby("segmento"):
    ids = grupo.index
    matriz_segmento = matriz.loc[ids]
    rango_svd = calcular_puntajes_svd(matriz_segmento, n_factores=40).rank(
        axis=1, pct=True
    )
    rango_popularidad = matriz_segmento.gt(0).sum(axis=0).rank(pct=True)
    puntajes.loc[ids] = 0.60 * rango_svd + 0.40 * rango_popularidad

scores = puntajes.to_numpy()
top3 = np.argsort(-scores, axis=1, kind="stable")[:, :3]
nombres = catalogo["product_name"].to_numpy()

# Marca los puntos de ruido de DBSCAN con los parámetros del CSV original.
arbol = KDTree(X_rfm)
es_nucleo = arbol.query_radius(X_rfm, r=0.4, count_only=True) >= 10
atipicos = np.ones(len(X_rfm), dtype=bool)
if es_nucleo.any():
    arbol_nucleos = KDTree(X_rfm[es_nucleo])
    atipicos = arbol_nucleos.query_radius(X_rfm, r=0.4, count_only=True) == 0

# Recomienda el producto mejor puntuado de una categoría aún no comprada.
categorias = pd.get_dummies(catalogo["category"]).to_numpy(dtype=np.int32)
compro_producto = matriz.gt(0).to_numpy(dtype=np.int32)
compro_categoria = (compro_producto @ categorias) > 0
categoria_nueva = (compro_categoria.astype(np.int32) @ categorias.T) == 0

scores_nuevos = np.where(categoria_nueva, scores, -np.inf)
mejor_nuevo = scores_nuevos.argmax(axis=1)

campanas = {
    "VIP": "Novedades y preventa exclusiva (sin descuentos agresivos)",
    "Regulares": "Venta cruzada de categorías que todavía no compra",
    "Ocasionales": "Cupón de segunda compra con los productos recomendados",
    "Inactivos": "Campaña de reactivación con recordatorio de su categoría favorita",
}
kpis = {
    "VIP": "Ticket promedio",
    "Regulares": "Variedad de productos",
    "Ocasionales": "Recurrencia",
    "Inactivos": "Recurrencia",
}

recomendaciones = clientes_gmm[["segmento", "prob_segmento"]].copy()
recomendaciones["atipico"] = atipicos
for columna in ["recencia_dias", "num_transacciones", "gasto_total"]:
    recomendaciones[columna] = clientes_gmm[columna]
for posicion in range(3):
    recomendaciones[f"producto_{posicion + 1}"] = nombres[top3[:, posicion]]

recomendaciones["producto_nueva_categoria"] = np.where(
    categoria_nueva.any(axis=1), nombres[mejor_nuevo], "—"
)
recomendaciones["campana"] = recomendaciones["segmento"].map(campanas)
recomendaciones["kpi_objetivo"] = recomendaciones["segmento"].map(kpis)

recomendaciones.to_excel(
    archivo,
    sheet_name="Recomendaciones",
    index_label="cliente_id",
    freeze_panes=(1, 1),
)

print(f"{len(recomendaciones):,} clientes exportados a {archivo}")
display(recomendaciones.head())

44,982 clientes exportados a recomendaciones_danustore_svd_80_20.xlsx


,segmento,prob_segmento,atipico,recencia_dias,num_transacciones,gasto_total,producto_1,producto_2,producto_3,producto_nueva_categoria,campana,kpi_objetivo
cliente_id,,,,,,,,,,,,
1,Regulares,0.968549,False,28,14,5778.12,Pants Product 55,Bags Product 68,Bottoms Product 17,—,Venta cruzada de categorías que todavía no compra,Variedad de productos
2,Regulares,0.968316,False,37,16,6972.74,Bags Product 68,Boots Product 107,Hats Product 83,—,Venta cruzada de categorías que todavía no compra,Variedad de productos
3,Regulares,0.678518,False,8,11,5492.28,Sneakers Product 97,Sneakers Product 96,Bottoms Product 17,—,Venta cruzada de categorías que todavía no compra,Variedad de productos
4,Ocasionales,0.985541,False,38,10,3115.28,Dresses Product 65,Bags Product 72,Hats Product 87,—,Cupón de segunda compra con los productos reco...,Recurrencia
5,Ocasionales,0.848927,False,29,13,3213.36,Dresses Product 65,Bags Product 68,Sneakers Product 127,—,Cupón de segunda compra con los productos reco...,Recurrencia
